Starting fresh from the original dim_employee

In [0]:
%sql
CREATE OR REPLACE TABLE ibm_hr.silver.dim_employee_scd3 AS
SELECT
  employee_id,
  Age, Gender, MaritalStatus,
  Department,
  CAST(NULL AS STRING) AS previous_department,
  JobRole, JobLevel, Education, EducationField, BusinessTravel, OverTime, Attrition
FROM ibm_hr.silver.dim_employee

Simulating the same change, employee 1 moving to Sales, using a MERGE that shifts the current value into previous_department before overwriting

In [0]:
%sql
MERGE INTO ibm_hr.silver.dim_employee_scd3 AS target
USING (SELECT 1 AS employee_id, 'Sales' AS new_department) AS source
ON target.employee_id = source.employee_id
WHEN MATCHED AND target.Department <> source.new_department THEN
  UPDATE SET
    target.previous_department = target.Department,
    target.Department = source.new_department

In [0]:
spark.sql("SELECT employee_id, Department, previous_department FROM ibm_hr.silver.dim_employee_scd3 WHERE employee_id = 1").show()